# Homework 1: Agentic RAG
In this homework, we build a RAG system from scratch and then make it agentic.

### Q1. How many lesson pages are in the dataset?
Download the markdown files from the `DataTalksClub/llm-zoomcamp` repository.

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda x: "/lessons/" in x
)

docs = [d.parse() for d in reader.read()]
print(f"Answer Q1 (Total lesson pages): {len(docs)}")


Answer Q1 (Total lesson pages): 72


### Q2. Indexing and searching
Index the documents with minsearch and find the most relevant file for our query.

In [3]:
import minsearch

index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(docs)

query = "How does the agentic loop keep calling the model until it stops?"
search_results = index.search(query)
print(f"Answer Q2 (First result filename): {search_results[0]['filename']}")


Answer Q2 (First result filename): 01-agentic-rag/lessons/14-agentic-loop.md


### Q3. RAG Setup
Adapt `RAGBase` from `rag_helper.py` to handle our schema and use Google's Gemini API to compute prompt tokens.

In [4]:
!curl -s https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py -o rag_helper.py

import os
from rag_helper import RAGBase
from dotenv import load_dotenv
from google import genai
import time

load_dotenv()
client = genai.Client()

class LessonRAG(RAGBase):
    def search(self, query, num_results=5):
        return self.index.search(query, num_results=num_results)

    def build_context(self, search_results):
        return '\n\n'.join([f"filename: {d['filename']}\ncontent: {d['content']}" for d in search_results])

    def llm(self, prompt):
        for _ in range(3):
            try:
                return client.models.generate_content(
                    model="gemini-3.1-flash-lite",
                    contents=prompt,
                    config=genai.types.GenerateContentConfig(
                        system_instruction=self.instructions,
                        temperature=0.0
                    )
                )
            except Exception:
                time.sleep(2)
        raise Exception("API timeout")

    def rag(self, query):
        res = self.search(query)
        prompt = self.build_prompt(query, res)
        resp = self.llm(prompt)
        return resp.text, resp.usage_metadata.prompt_token_count

rag_sys = LessonRAG(index=index, llm_client=client)
_, tokens_q3 = rag_sys.rag(query)
print(f"Answer Q3 (Prompt tokens): {tokens_q3}")


Answer Q3 (Prompt tokens): 7958


### Q4. Chunking
Divide the documents into smaller chunks using a sliding window to improve retrieval precision.

In [5]:
from gitsource import chunk_documents
chunked_docs = chunk_documents(docs, size=2000, step=1000)
print(f"Answer Q4 (Total chunks): {len(chunked_docs)}")


Answer Q4 (Total chunks): 295


### Q5. RAG with Chunking
Measure token reduction after chunking the documents.

In [6]:
index_chunked = minsearch.Index(text_fields=["content"], keyword_fields=["filename"])
index_chunked.fit(chunked_docs)

rag_sys_chunked = LessonRAG(index=index_chunked, llm_client=client)
_, tokens_q5 = rag_sys_chunked.rag(query)
print(f"Answer Q5:")
print(f" - Prompt tokens with chunking: {tokens_q5}")
print(f" - Reduction Ratio: {tokens_q3 / tokens_q5:.2f}x")


Answer Q5:
 - Prompt tokens with chunking: 2611
 - Reduction Ratio: 3.05x


### Q6. Agentic RAG
Build an Agentic loop where the LLM decides when to call the `search` tool.

In [7]:
from google.genai import types

search_count = 0

def search_faq(search_query: str) -> str:
    global search_count
    search_count += 1
    res = index_chunked.search(search_query)
    return '\n\n'.join([f"filename: {d['filename']}\ncontent: {d['content']}" for d in res])

instructions = "You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."

chat = client.chats.create(
    model="gemini-3.1-flash-lite",
    config=types.GenerateContentConfig(
        tools=[search_faq], 
        temperature=0.0,
        system_instruction=instructions
    )
)

agent_query = "How does the agentic loop work, and how is it different from plain RAG?"

for i in range(10):
    resp = chat.send_message(agent_query if i == 0 else "")
    if resp.function_calls:
        for fc in resp.function_calls:
            if fc.name == "search_faq":
                out = search_faq(**fc.args)
                chat.send_message(out)
    else:
        break

print(f"Answer Q6 (Total search calls): {search_count}")


Answer Q6 (Total search calls): 2
